# U-Net Based Semantic Segmentation
This model is trainned on the Wildscene dataset (2D part) built by Vidanapathirana, Kavisha et. al.<br>
dataset: https://data.csiro.au/collection/csiro%3A61541v2 <br>
dataset details provided by author: https://csiro-robotics.github.io/WildScenes/  <br>

The dataset is not included in the repo. <br>
for any problem, email: Diwen.Ye@student.unsw.edu.au <br>

# Download trained model
https://drive.google.com/drive/folders/1wrF_j-CJrvl_JjCUoOsbnxEnHBF6YYNa?usp=sharing 
### Place the downloaded model my_checkpoint8c.pth.tar to /model folder

In [76]:
import torch
import torch.nn as nn
from torch.nn.functional import relu
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from torchvision import models
import torchvision.transforms.functional as TF
print("torch version: "+torch.__version__+" | torch.cuda's access to gpu: "+ str(torch.cuda.is_available()) + " | cuda version: " + str(torch.version.cuda))

torch version: 2.3.0 | torch.cuda's access to gpu: True | cuda version: 12.1


In [77]:
torch.zeros(1).cuda()

tensor([0.], device='cuda:0')

In [78]:
from PIL import Image #to read image
import os #to extract filenames
import cv2
import csv
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import random

## IndexLabel
IndexLabel are targeted outputs with value 0~18 (as index of each class), it is a 3-channeled `1512*2016 (3:4), (1512, 2016, 3)` picture, each channel share the same value, when converting to grayscale, it will still be the index value (grayscale will take channelwise average). 

# Folder structure
UNet19channel (the project folder, dataset excluded)
>model <br>
>saved_images <br>
>>epoch** (eg. epoch8a, epoch8b, epoch8c, number related to which epoch of training loop (8th for this script), a b c means which dataset's data have been covered in the dataset v01 v02 v03) <br>
>jaccard_scores.csv <br>
>Unet19channel.ipynb <br>

V-01 (the folder for dataset, V-01 V-02 V-03 all needed here)<br>
>image <br>
>indexLabel <br>
>label (not used here) <br>

V-02 <br>
>...

V-03 <br>
>...

## Path to store the model

In [79]:
model_path = "model/"

# Create list of input and target
All the training files are named by their timestamp, eg. "1623377790-818434554.png" <br>
Input Images and Output Labels indexLabels share the same filename. <br>

In [80]:
#path to input files
path_in1 = "../V-01/image"
#path to targeted output mask files
path_target1= "../V-01/indexLabel"

path_in2 = "../V-02/image"
path_target2= "../V-02/indexLabel"
path_in3 = "../V-03/image"
path_target3= "../V-03/indexLabel"

#path to image save files
path_in_demo = "../V-01/image"
#path to targeted output mask files
path_target_demo= "../V-01/indexLabel"

In [81]:
#function to test if dataset is correctly placed
def generate_file_list(mypath):
    files = [f for f in os.listdir(mypath) if os.path.isfile(os.path.join(mypath, f))]
    return files

In [82]:
#in dataset formation, taking target filenames as base
in_names1 = generate_file_list(path_in1)
target_names1 = generate_file_list(path_target1)
in_names2 = generate_file_list(path_in2)
target_names2 = generate_file_list(path_target2)
in_names3 = generate_file_list(path_in3)
target_names3 = generate_file_list(path_target3)

in_names_demo = generate_file_list(path_in_demo)
target_names_demo = generate_file_list(path_target_demo)

print("Input_files1: "+ str(len(in_names1)) + " | Target_files1: " + str(len(target_names1)))
print("Input_files2: "+ str(len(in_names2)) + " | Target_files2: " + str(len(target_names2)))
print("Input_files3: "+ str(len(in_names3)) + " | Target_files3: " + str(len(target_names3)))

Input_files1: 743 | Target_files1: 743
Input_files2: 833 | Target_files2: 833
Input_files3: 1845 | Target_files3: 1845


In [83]:
idx_split1 = int(len(target_names1) * 0.9)
training_set1 = target_names1[:idx_split1]
testing_set1 = target_names1[idx_split1:]

idx_split2 = int(len(target_names2) * 0.9)
training_set2 = target_names2[:idx_split2]
testing_set2 = target_names2[idx_split2:]

idx_split3 = int(len(target_names3) * 0.9)
training_set3 = target_names3[:idx_split3]
testing_set3 = target_names3[idx_split3:]

training_set = training_set1+training_set2+training_set3
testing_set = testing_set1+testing_set2+testing_set3

idx_split_demo = int(len(target_names_demo) * 0.9)
testing_set_demo = target_names_demo[idx_split_demo:]

print("Input_files: "+ str(len(training_set)) + " | Output_files: " + str(len(testing_set)))
print("demo_files: " + str(len(testing_set_demo)))

Input_files: 3077 | Output_files: 344
demo_files: 75


# U-Net Model

In [84]:
class DoubleConv(nn.Module):
    def __init__(self, in_channel, out_channel):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channel, out_channel, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channel, out_channel, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class Unet(nn.Module):
    def __init__(self, in_channel=3, out_channel=1, features=None):
        super(Unet, self).__init__()
        if features is None:
            features = [64, 128, 256, 512]

        self.u_sample = nn.ModuleList()
        self.d_sample = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        for feature in features:
            self.d_sample.append(DoubleConv(in_channel, feature))
            in_channel = feature

        for feature in reversed(features):
            self.u_sample.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=3, stride=1, padding=1))
            self.u_sample.append(DoubleConv(in_channel=feature*2, out_channel=feature))

        self.bottle_neck = DoubleConv(features[-1], features[-1]*2)
        self.final_conv = nn.Conv2d(features[0], out_channel, kernel_size=1)

    def forward(self, x):
        skip_connections = []    # to retain features

        for down in self.d_sample:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottle_neck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.u_sample), 2):
            x = self.u_sample[idx](x)

            skip_connection = skip_connections[idx // 2]
            if skip_connection.shape != x.shape:
                x = TF.resize(x, size=skip_connection.shape[2:])

            concat = torch.cat((x, skip_connection), dim=1)
            x = self.u_sample[idx + 1](concat)
        return nn.functional.softmax(self.final_conv(x), dim=1) #apply softmax along the "channel" dimension, which means for certain [batch,:,height,weight] it should sum to 1

# Train the model
## Hyper-parameters

In [85]:
#Hyperparameters
LEARNING_RATE = 1e-4
BATCH_SIZE = 8     # TODO: adjust based on model used and CROPPED_INPUT_SIZE
NUM_EPOCHS = 8
NUM_WORKERS = 0
PIN_MEMORY = True
DEVICE = 'cuda' #if torch.cuda.is_available() else 'cpu'
                    #cpu-only mode is not an option here as it can't afford the vRAM consumption of unet
CLASS_NUM = 19
IMG_H = 1512
IMG_W = 2016
#plan: resize the image into approprate size of resolution then crop the graph for approprate size
#Fixed resize + Random Crop
#cloud and sky is easier to identify as they have different Hue, the difficult part are
#the ground, road and trees, in the middle to bottom part.
#currently its resize to 864*1152, crop to 432*576
#with less blurring and enough number of output classes in one crop
INPUT_H = 432 #need to be product of 16
INPUT_W = 576

LABEL_LIST=[
        [0, 0, 0],
        [230, 25, 75],
        [60, 180, 75],
        [255, 225, 25],
        [0, 130, 200],
        [145, 30, 180],
        [70, 240, 240],
        [240, 50, 230],
        [210, 245, 60],
        [230, 25, 75],
        [0, 128, 128],
        [170, 110, 40],
        [255, 250, 200],
        [128, 0, 0],
        [170, 255, 195],
        [128, 128, 0],
        [250, 190, 190],
        [0, 0, 128],
        [128, 128, 128],
    ]
LABEL_MAP=np.array(LABEL_LIST, dtype=np.float32)

train_losses = []
test_acc = []
test_jaccard = []

### reshape the output
U-NET output format is [channel, height, width], need to be reshaped to [height, width, channel]. <br>
Then need to reshape the 19 channel output into RGB image <br>
Meanwhile, for training steps, prediction result shall not be touched so that the gradients can be preserved for back propergation <br>
In this case, Targets need to be changed <br>

In [86]:
def reshape_output_channel(masks): #input tensor format, output numpy format
    output = np.empty((INPUT_H, INPUT_W, CLASS_NUM), dtype=np.float32)
    for i in range(CLASS_NUM):
        mask_cln = masks.clone().detach()
        output[:, :, i] = mask_cln.cpu().detach().numpy()[i].astype(np.float32) #assign each channel of mask of binary map
                                                                    #detach() removes gradiants, so it can only be done on copy instead of training output
    return output
def reshape_output_batch(batches): #input tensor, output tensor
    #re-organise size to [batch, h, w, channel]
    np_batches = np.empty((batches.size(dim=0),batches.size(dim=2),batches.size(dim=3),batches.size(dim=1)), dtype=np.float32) 
    for i in range(len(batches)):
        np_batches[i] = reshape_output_channel(batches[i])
    return torch.from_numpy(np_batches).float().to(device=DEVICE)

In [87]:
def reshape_target_channel(masks): #input tensor format [h, w, channel], output numpy format [channel, h, w] (align with unet output)
    #output = np.empty((CLASS_NUM, INPUT_H, INPUT_W), dtype=np.float32)
    h = masks.size(dim=0) #channel, h, w
    w = masks.size(dim=1)
    channel = masks.size(dim=2)
    output = np.empty((channel, h, w), dtype=np.float32)
    for i in range(CLASS_NUM):
        mask_cln = masks.clone().detach()
        output[i] = mask_cln.cpu().detach().numpy()[:,:,i].astype(np.float32) #assign each channel of mask of binary map
                                                                    #detach() removes gradiants, so it can only be done on copy instead of training output
    return output
def reshape_target_batches(batches):
    #re-organise size to [batch, channel, h, w]
    np_batches = np.empty((batches.size(dim=0),batches.size(dim=3),batches.size(dim=1),batches.size(dim=2)), dtype=np.float32)
    for i in range(len(batches)):
        np_batches[i] = reshape_target_channel(batches[i])
    return torch.from_numpy(np_batches).float().to(device=DEVICE)

In [88]:
def masks_to_img(masks): #input tensor output numpy
    h = masks.size(dim=1) #channel, h, w
    w = masks.size(dim=2)
    #img = np.zeros((INPUT_H, INPUT_W, 3), dtype=np.float32)
    img = np.zeros((h, w, 3), dtype=np.float32)
    masks_np = masks.cpu().detach().numpy()
    mask_map = np.argmax(masks_np, axis = 0) #map the class with maximum potential for each pixel
    for i in range(CLASS_NUM):
        #img[masks_np[i,:,:]>0.5]=LABEL_MAP[i] #prediction masks is in the format of [channel, h, w]
        img[mask_map==i]=LABEL_MAP[i] #map the pixel class with corresponding color
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img
def batch_to_img(batches): #input tensor batch output numpy batch
    #np_batches = np.empty((batches.size(dim=0),INPUT_H,INPUT_W,3), dtype=np.float32)
    np_batches = np.empty((batches.size(dim=0),batches.size(dim=2),batches.size(dim=3),3), dtype=np.float32)
    for i in range(len(batches)):
        np_batches[i] = masks_to_img(batches[i])
    return np_batches

## create a new model instance (CREATE FIRST, THEN LOAD THE CHECKPOINT TO THIS MODEL)
Can choose to load the model instead (at the end of notebook)

In [89]:
MODEL = Unet(
        out_channel=CLASS_NUM
    ).to(DEVICE)

## Dataset structure

In [90]:
#create dataset class, which is the class that we use to extract img&mask matrix from path
#I added an extra target_names list, because I extracted the training set and validation set manually
class WildSceneDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None, target_names=[]):
        #img is default but the mask need to be rescaled to 18 channels from indexLabels
        #input and target files share the same format and filename
        #here use target files as list of data, to avoid not segmented inputs cases
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.imgs = target_names if not len(target_names) == 0 else generate_file_list(mask_dir)
    def __len__(self):
        return len(self.imgs)
    def wrap_to_19channel(self, target):
        mask = np.zeros((1512, 2016, 19), dtype=np.float32)
        for i in range(19):
            mask[:, :, i] = (target == i).astype(np.float32) #assign each channel of mask of binary map
        return mask
    def __getitem__(self, index): #get path of specific item in index
        img_path = os.path.join(self.img_dir, self.imgs[index])
        mask_path = os.path.join(self.mask_dir, self.imgs[index]) 
                                                #self.imgs[index].replace(".jpg", "_mask.png") is a trick to reformat filename format
        img = np.array(Image.open(img_path).convert("RGB")) # shape is row*col*channel (1512, 2016, 3)
        target = np.array(Image.open(mask_path).convert("L")) #grayscale
        mask = self.wrap_to_19channel(target)
        
        if self.transform is not None:
            augmentations = self.transform(image=img, mask=mask)
            img = augmentations["image"]
            mask = augmentations["mask"]
        return img, mask

## utils functions 
### function to save and load model

In [91]:
def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state,filename)
def load_checkpoint(checkpoint, model):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])
    print("=> checkpoint loaded")

### function to set up the training data and validation data loader

In [92]:
def get_loaders(
    train_dir,
    train_maskdir,
    val_dir,
    val_mask_dir,
    batch_size,
    train_transform,
    val_transform,
    num_workers=2,
    pin_memory=True,
    train_list=[], #for this project specifically, as I get train and val set from same folder
    val_list=[] #for this project specifically
):
    train_ds = WildSceneDataset(
        img_dir=train_dir, 
        mask_dir=train_maskdir, 
        transform=train_transform, 
        target_names=train_list #if don't wanna split training set and validation set in separate folder
                                 #can manually set the list of filename
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
        shuffle=True,      #turn that to false if the model is time-sequence-sensitive
                            #such as SNN, RNN etc
    )
    val_ds = WildSceneDataset(
        img_dir=val_dir, 
        mask_dir=val_mask_dir, 
        transform=val_transform, 
        target_names=val_list #manually set the list of filename
    )
    val_loader=DataLoader(
        val_ds,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
        shuffle=False,   #for validation, forming a sequential video demo
    )
    return train_loader, val_loader

## Training function for 1 epoch
Training loop in function `training`, which will do 1 epoch of training

In [93]:
def training(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)

    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device=DEVICE)
        targets = targets.float().to(device=DEVICE)
        # forward
        with torch.cuda.amp.autocast():
            predictions = model(data)
            loss = loss_fn(predictions, reshape_target_batches(targets)) #predictions shall be left as it is with calculated gradiants
        # back propergation
        optimizer.zero_grad() #initialize the gradiant
        scaler.scale(loss).backward()
        scaler.step(optimizer) #do one step of back propergation
        scaler.update()
        #update tqdm loop
        loop.set_postfix(loss=loss.item())

set up the preprocessing transform method for training set and validation set <br>
along with the loss function, data loader and gradiant scaler

In [94]:
#for training set
#normalize will only apply on input set, other will apply on both input and target
train_transform = A.Compose(
    [ 
        A.Resize(height = 2*INPUT_H, width= 2*INPUT_W),
        A.RandomCrop(height=INPUT_H, width=INPUT_W),
        A.Rotate(limit = 35, p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.1),
        A.Normalize(
            mean=[0.0, 0.0, 0.0],
            std=[1.0, 1.0, 1.0],
            max_pixel_value=255.0,
        ),
        ToTensorV2(),
    ],
)
#for validation sets, we're expecting similar scale
val_transform = A.Compose(
    [
        A.Resize(height = 2*INPUT_H, width= 2*INPUT_W),
        A.RandomCrop(height=INPUT_H, width=INPUT_W),
        A.Normalize(
            mean=[0.0, 0.0, 0.0],
            std=[1.0, 1.0, 1.0],
            max_pixel_value=255.0,
        ),
        ToTensorV2(),
    ],
)

#loss_fn = nn.BCEWithLogitsLoss() #for output without sigmoid()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(MODEL.parameters(), lr=LEARNING_RATE)

train_loader1, val_loader1 = get_loaders(
    path_in1,
    path_target1,
    path_in1,  #the validation set comes from the same folder with training set, manual list splitting has already been done before
    path_target1,
    BATCH_SIZE,
    train_transform,
    val_transform,
    NUM_WORKERS,
    PIN_MEMORY,
    train_list= training_set1, #manual list splitting is used here, if reading training and testing data from different directory, leave blank
    val_list= testing_set1
)
train_loader2, val_loader2 = get_loaders(
    path_in2,
    path_target2,
    path_in2,  #the validation set comes from the same folder with training set, manual list splitting has already been done before
    path_target2,
    BATCH_SIZE,
    train_transform,
    val_transform,
    NUM_WORKERS,
    PIN_MEMORY,
    train_list= training_set2, #manual list splitting is used here, if reading training and testing data from different directory, leave blank
    val_list= testing_set2
)
train_loader3, val_loader3 = get_loaders(
    path_in3,
    path_target3,
    path_in3,  #the validation set comes from the same folder with training set, manual list splitting has already been done before
    path_target3,
    BATCH_SIZE,
    train_transform,
    val_transform,
    NUM_WORKERS,
    PIN_MEMORY,
    train_list= training_set3, #manual list splitting is used here, if reading training and testing data from different directory, leave blank
    val_list= testing_set3
)

scaler = torch.cuda.amp.GradScaler()


### Set up loader for demo image save

In [95]:
#for the formal output, it is designed for convolution of one whole image without crop, resized at same proportion
img_save_transform = A.Compose(
    [
        A.Resize(height = 2*INPUT_H, width= 2*INPUT_W),
        A.Normalize(
            mean=[0.0, 0.0, 0.0],
            std=[1.0, 1.0, 1.0],
            max_pixel_value=255.0,
        ),
        ToTensorV2(),
    ],
)
img_save_ds = WildSceneDataset(
    img_dir=path_in_demo, 
    mask_dir=path_target_demo, 
    transform=img_save_transform, 
    target_names=testing_set_demo #manually set the list of filename
)
img_save_loader=DataLoader(
    img_save_ds,
    batch_size=1,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    shuffle=False,   #for validation, forming a sequential video demo
)

### function to check the model accuracy

In [96]:
def check_accuracy(loader, model, epoch, device="cuda"):
    loop = tqdm(loader)
    num_correct = 0
    num_pixels = 0
    num_true_positive = 0
    num_positive = 0
    num_true_negative = 0
    jaccard_scores = [0] * CLASS_NUM
    class_counts = [0] * CLASS_NUM
    model.eval() #set to eval mode
    with torch.no_grad(): #in evaluation step, we don't calculate gradiant of each weight
        for x,y in loop: # x input, y output
            x = x.to(device)
            y = y.to(device)
            preds = model(x) #if model output didn't wrap sigmoid (wrap from 0 to 1), do it here    
            preds_np = preds.cpu().detach().numpy() 
            preds_bool = (preds > 0.5) #decision model, scale probability to 0 and 1
            preds = preds_bool.float() #preds are wrapped to 0.0 and 1.0 in float

            preds_np = preds.cpu().detach().numpy() 
            preds_bool_np = preds_bool.cpu().detach().numpy()
            y_reshaped = reshape_target_batches(y).cpu().detach().numpy()
            num_correct += (preds_np == y_reshaped).astype(np.float32).sum()
            num_pixels += torch.numel(preds)
            num_true_positive += ((preds_np == y_reshaped) & preds_bool_np).astype(np.float32).sum() #number of true positive
            num_positive += y_reshaped.sum() #the number of ground truth positive
                                            #true negative is calculated after the loop
            

            for channel in range(CLASS_NUM):
                true_cls = (preds_np[:,channel,:,:] == y_reshaped[:,channel,:,:]) #bool
                intersection = (preds_bool_np[:,channel,:,:] & true_cls).astype(np.float32).sum() 
                union = (preds_bool_np[:,channel,:,:] | (y_reshaped[:,channel,:,:]>0.5)).astype(np.float32).sum()
                if union == 0:
                    jaccard_scores[channel] += 0
                else:
                    jaccard_scores[channel] += intersection / union
                class_counts[channel] += 1
    num_true_negative = num_correct - num_true_positive
    accuracy = round(float(num_correct / num_pixels), 4)
    jaccard_per_class = [jaccard_scores[i] / class_counts[i] for i in range(CLASS_NUM)] 
    miou = round(sum(jaccard_per_class) / CLASS_NUM, 4)
    with open('jaccard_scores.csv', mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([epoch] + jaccard_per_class)
            print('Jaccard Scores are saved to jaccard_scores.csv')
    print(f'Got {num_correct} / {num_pixels} with acc {num_correct / num_pixels * 100 :.2f}%, TPR={num_true_positive / num_positive * 100 :.2f}%, TNR={num_true_negative / (num_pixels-num_positive) * 100 :.2f}%')
    print(f'Mean IoU: {miou}')
    
    model.train() #set back to train
    return accuracy, jaccard_per_class, miou


In [97]:
def save_predictions_as_imgs(
    loader, model, folder="saved_images/", device="cuda"
):
    model.eval()
    for idx, (x, y) in enumerate(loader):
        x = x.to(device=device)
        with torch.no_grad():
            preds = model(x)   #if not include sigmoid() in neural network model output, wrap torch.sigmoid() here
            preds = preds.float() 
        imgs_pred = batch_to_img(preds)
        imgs_y = batch_to_img(reshape_target_batches(y))
        for i in range(len(imgs_pred)):
            cv2.imwrite(f"{folder}{idx}.png", imgs_y[i])
            cv2.imwrite(f"{folder}/{idx}_pred.png", imgs_pred[i])
    model.train()

# (Optional) Load the model

In [98]:
load_checkpoint(torch.load(model_path+"my_checkpoint8c.pth.tar"), MODEL)

=> Loading checkpoint
=> checkpoint loaded


## Training Loop (LOAD MODEL BEFORE TRAINING)
In each Epoch: <br>
The model will be trained on all images in training set - `training()` <br>
The trained model will be saved as checkpoint - `save_checkpoint()` <br>
The trained model will be evaluated on validate set (testing set) and write to jaccard_scores.csv as one of the testing score - `check_accuracy()` <br>
The prediction of validation set will be printed in folder 'saved_images' - `save_predictions_as_imgs()` <br>

Training log: <br>
`epoch 1`: V01 dataset, the CrossEntropyLoss starts from 2.94 (which is approximately to log(19) for 19 classes) as expected <br>
        the loss is decreasing at a rate of 0.01/batch till around 2.3 <br>
        learning rate at 1e-3 <br>
`epoch 2`: V01 dataset, the loss is fluctuating around 2.3, 
        It is able to capture some feature, especially about the foliage and trunk <br>
        the test segmentation is printed in corresponding folder <br>
`epoch 3a` (dropped): V01 dataset, debug trial 1: try to reduce learning rate to 1e-4, loss is still fluctuating, which means it's more like to be trapped in local minima or simply because the speed of convergence is too slow, need to increase the learning rate <br>
        **starts to ignore the trunks** <br>
`epoch 3b`: V02 dataset, restarting from checkpoint after epoch2, with 1e-3 learning rate <br>
        the bush and dirt class are not successfully identified, need more information input. <br>
`epoch 3c`: V02 dataset, expanding dataset, from checkpoint after epoch 3b, with 5e-3 learning rate and removed crop in data augmentation <br>
        **error detected:** out of memory, restricting the input size is important, random crop still need to be included in the training set. <br>
        continue with crop on. <br>
        **error detected:** 5e-3 learning rate have output as all NaN (potentially overflow) reduce the learning rate to 2e-3 <br>
        output ignores the trunk, probably caused by the high learning rate, reduce learning rate to 1e-3 <br>
        **starts to ignore the trunks** <br>
`epoch 3d`: V03 dataset, expanding dataset, from checkpoint 3b, learning rate 5e-4 <br>
        training loss starts to increase, it indicates that the learning rate might be too high <br>
        pause training and test print the prediction, starts to ignore the trunks <br>
        change learning rate to 1e-5, the increase stops but the fluctuating range didnot decrease. It indicates that the cause may be related to the batch size, model structure, biased sample or simply noise. <br>
        can try to increase the learning rate back to 1e-4 in next batch <br>
`epoch 4~8 abc`: training on combination of V01's training set, V02's training set and V03's training set. At learning rate of 1e-4 <br>

>epoch4: loss=[2.16 2.32 2.49] mIoU = [0.0321 0.0419 0.0292] |val_v03 accuracy = 95.21% `TPR = 51.19%` TNR = 97.66% (iterating through V01 V02 V03) <br>
>epoch5: loss=[2.23 2.33 2.28] mIoU = [0.0381 0.0430 0.0385] |val_v03 accuracy = 96.97% `TPR = 67.24%` TNR = 98.62%<br>
>epoch6: loss=[2.32 2.15 2.35] mIoU = [0.0378 0.0431 0.0427] |val_v03 accuracy = 97.52% `TPR = 75.90%` TNR = 98.72%<br>
>epoch7: loss=[2.31 2.20 2.23] mIoU = [0.0444 0.0458 0.0434] |val_v03 accuracy = 97.83% `TPR = 79.09%` TNR = 98.87%<br>
>epoch8: loss=[2.24 2.28 2.30] mIoU = [0.0434 0.0463 0.0434] |val_v03 accuracy = 97.79% `TPR = 78.83%` TNR = 98.85%<br>

        


In [46]:
#training loop
for epoch in range(3,NUM_EPOCHS):
    # -----------for V01 dataset ---------------------
    print("---------------training start: epoch" + str(epoch+1) + "|V01---------------")
    training(train_loader1, MODEL, optimizer, loss_fn, scaler)
    #save model
    checkpoint = {
        "state_dict": MODEL.state_dict(),
        "optimizer": optimizer.state_dict(),
    }
    save_checkpoint(checkpoint, model_path+"my_checkpoint"+str(epoch+1)+"a.pth.tar")
    #check accuracy
    check_accuracy(val_loader1, MODEL, epoch, device=DEVICE)
    
    # -----------for V02 dataset ---------------------
    print("---------------training start: epoch" + str(epoch+1) + "|V02---------------")
    training(train_loader2, MODEL, optimizer, loss_fn, scaler)
    checkpoint = {
        "state_dict": MODEL.state_dict(),
        "optimizer": optimizer.state_dict(),
    }
    save_checkpoint(checkpoint, model_path+"my_checkpoint"+str(epoch+1)+"b.pth.tar")
    check_accuracy(val_loader2, MODEL, epoch, device=DEVICE)

    # -----------for V03 dataset ---------------------
    print("---------------training start: epoch" + str(epoch+1) + "|V03---------------")
    training(train_loader3, MODEL, optimizer, loss_fn, scaler)
    #save model
    checkpoint = {
        "state_dict": MODEL.state_dict(),
        "optimizer": optimizer.state_dict(),
    }
    save_checkpoint(checkpoint, model_path+"my_checkpoint"+str(epoch+1)+"c.pth.tar")
    check_accuracy(val_loader3, MODEL, epoch, device=DEVICE)
    
    
    #print examples
    #save_predictions_as_imgs(
    #    val_loader, MODEL, folder="saved_images/epoch"+str(epoch)+"/", device=DEVICE
    #)
     

---------------training start: epoch4|V01---------------


100%|███████████████████████████████████████████████████████████████████████| 84/84 [17:05<00:00, 12.21s/it, loss=2.16]


=> Saving checkpoint


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [02:56<00:00, 17.62s/it]


Jaccard Scores are saved to jaccard_scores.csv
Got 339950271.0 / 354585600 with acc 95.87%, TPR=60.76%, TNR=97.82%
Mean IoU: 0.0321
---------------training start: epoch4|V02---------------


...

Jaccard Scores are saved to jaccard_scores.csv
Got 391239436.0 / 397135872 with acc 98.52%, TPR=85.84%, TNR=99.22%
Mean IoU: 0.0463
---------------training start: epoch8|V03---------------


100%|██████████████████████████████████████████████████████████████████████| 208/208 [48:26<00:00, 13.97s/it, loss=2.3]


=> Saving checkpoint


100%|██████████████████████████████████████████████████████████████████████████████████| 24/24 [07:37<00:00, 19.08s/it]

Jaccard Scores are saved to jaccard_scores.csv
Got 855334820.0 / 874644480 with acc 97.79%, TPR=78.83%, TNR=98.85%
Mean IoU: 0.0434


In [59]:
#check accuracy
check_accuracy(val_loader3, MODEL, 8, device=DEVICE)

100%|██████████████████████████████████████████████████████████████████████████████████| 24/24 [06:49<00:00, 17.05s/it]

Jaccard Scores are saved to jaccard_scores.csv
Got 855089508.0 / 874644480 with acc 97.76%, TPR=78.57%, TNR=98.83%
Mean IoU: 0.1411


(0.9776,
 [0.0,
  0.0,
  0.7738045814136664,
  0.0,
  0.0,
  0.0,
  0.0,
  0.4969532676041126,
  0.7230029292404652,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.1799618971417658,
  0.5067458053429922],
 0.1411)

## Save the prediction image for demo
**Manually** create the corresponding epoch** folder in the saved_images first, (eg. epoch8c here)

In [99]:
save_predictions_as_imgs(
        img_save_loader, MODEL, folder="saved_images/epoch"+str(8)+"c/", device=DEVICE
    )

KeyboardInterrupt: 